# 06 · Разведка: перенос методов на новые типы требований

**Вопрос (H4: условное vs глобальное).** Steering, гейт, TACT и LoRA выбраны на val в 04 и учились на
требованиях «срок / количество / бюджет / наличие». Переносится ли их эффект на требования, которых не
было в train, — **рейтинг** и **гарантия**? Для контроля вторая половина набора — знакомые типы на свежем
seed, в том же формате ответа (все поля, включая `rating` и `warranty_months`).

**Как читать.** Разница прироста метода между половинами seen и unseen — его переносимость (один из
восьми критериев 2604.14090). Методы заморожены на val, на этом наборе ничего не подбирается; test-сплит
не используется. Результат **разведочный**.

> Всё до `fit` берётся из кэша 04 (те же конфиг вычислений, данные и модель).

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [2]:
cfg = ExperimentConfig.make(
    name="transfer-qwen3-1.7b", hypothesis="explore", notebook="notebooks/06_transfer.ipynb",
    notes="Перенос выбранных на val методов на новые типы требований (рейтинг, гарантия).",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "strong_prompt", "restate", "explicit_state",
                         "steering", "steering_gated", "tact", "lora"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "transfer.n_tasks": 100, "transfer.seed": 101,
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")

/Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[00:51:22] run 20261020-005120_transfer-qwen3-1.7b_c2afcb  ->  results/runs/20261020-005120_transfer-qwen3-1.7b_c2afcb
[00:51:22] data: {'train': 79, 'val': 31, 'test': 50}  types={'invariant': 41, 'changing': 112, 'stress': 7}  positions={'early': 114, 'late': 46}


baseline/train:   0%|          | 0/158 [00:00<?, ?it/s]

[00:51:22] loading model Qwen/Qwen3-1.7B …


W1020 00:51:25.786000 14125 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

... (249 outputs left out)

episodes tact/val:   0%|          | 0/58 [00:00<?, ?it/s]

episodes lora/val:   0%|          | 0/58 [00:00<?, ?it/s]

[00:51:42] val: baseline 0.621; strong_prompt 0.569; restate 0.638; explicit_state 0.552; steering 0.655; steering_gated 0.638; tact 0.621; lora 0.948


,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.621,0.621,0.526,0.4,NaN,,NaN,NaN,0.586,NaN,,0.50,831,10.0,400.6
1,strong_prompt,0.569,0.569,0.474,0.3,-0.052,"[-0.138, +0.000]",0.0,3.0,0.569,-0.017,"[-0.052, +0.000]",1.00,885,10.0,291.8
2,restate,0.638,0.638,0.526,0.4,0.017,"[-0.069, +0.103]",4.0,3.0,0.586,0.000,"[-0.086, +0.086]",0.50,1765,11.0,1596.1
3,explicit_state,0.552,0.552,0.474,0.1,-0.069,"[-0.207, +0.069]",6.0,10.0,0.466,-0.121,"[-0.276, +0.034]",0.25,1932,11.0,3443.7
4,steering,0.655,0.655,0.632,0.4,0.034,"[+0.000, +0.086]",2.0,0.0,0.603,0.017,"[+0.000, +0.052]",0.50,831,10.0,410.2
5,steering_gated,0.638,0.638,0.579,0.4,0.017,"[+0.000, +0.052]",1.0,0.0,0.603,0.017,"[+0.000, +0.052]",0.50,831,10.0,408.6
6,tact,0.621,0.621,0.526,0.4,0.000,"[+0.000, +0.000]",0.0,0.0,0.586,0.000,"[+0.000, +0.000]",0.50,831,10.0,369.6
7,lora,0.948,0.948,1.000,0.8,0.328,"[+0.190, +0.466]",19.0,0.0,0.828,0.241,"[+0.121, +0.379]",0.00,831,10.0,299.0


### Перенос: знакомые типы требований (seen) и новые (unseen)

In [3]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))

baseline/transfer:seen:   0%|          | 0/100 [00:00<?, ?it/s]

baseline/transfer:seen:   1%|          | 1/100 [00:01<02:06,  1.28s/it]

baseline/transfer:seen:   2%|▏         | 2/100 [00:01<01:19,  1.24it/s]

baseline/transfer:seen:   3%|▎         | 3/100 [00:02<01:03,  1.52it/s]

baseline/transfer:seen:   4%|▍         | 4/100 [00:02<00:58,  1.65it/s]

baseline/transfer:seen:   5%|▌         | 5/100 [00:03<01:12,  1.31it/s]

... (2,221 outputs left out)

**seen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.53,0.53,0.244,0.333,NaN,,NaN,NaN,0.45,NaN,,None,859,10.0,300.7
1,strong_prompt,0.46,0.46,0.195,0.333,-0.07,"[-0.160, +0.010]",3.0,10.0,NaN,NaN,,None,913,10.0,303.7
2,restate,0.59,0.59,0.268,0.444,0.06,"[-0.030, +0.160]",13.0,7.0,NaN,NaN,,None,1850,11.0,2308.7
3,explicit_state,0.61,0.61,0.268,0.333,0.08,"[-0.010, +0.170]",22.0,14.0,NaN,NaN,,None,2018,11.0,4293.9
4,steering,0.52,0.52,0.220,0.444,-0.01,"[-0.090, +0.080]",6.0,7.0,0.41,-0.04,"[-0.120, +0.050]",None,859,10.0,300.0
5,steering_gated,0.53,0.53,0.220,0.333,0.00,"[-0.070, +0.070]",5.0,5.0,NaN,NaN,,None,859,10.0,300.3
6,tact,0.53,0.53,0.244,0.333,0.00,"[+0.000, +0.000]",0.0,0.0,NaN,NaN,,None,859,10.0,320.6
7,lora,0.90,0.90,0.756,1.000,0.37,"[+0.260, +0.480]",37.0,0.0,0.77,0.32,"[+0.190, +0.440]",None,859,10.0,317.7


**unseen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.54,0.54,0.147,0.562,NaN,,NaN,NaN,0.49,NaN,,None,866,10.0,310.2
1,strong_prompt,0.47,0.47,0.206,0.438,-0.07,"[-0.150, +0.000]",2.0,9.0,NaN,NaN,,None,920,10.0,319.1
2,restate,0.47,0.47,0.147,0.438,-0.07,"[-0.160, +0.020]",5.0,12.0,NaN,NaN,,None,1869,11.0,2350.9
3,explicit_state,0.85,0.85,0.618,0.938,0.31,"[+0.200, +0.420]",34.0,3.0,NaN,NaN,,None,2047,11.0,4374.2
4,steering,0.52,0.52,0.206,0.562,-0.02,"[-0.110, +0.060]",8.0,10.0,0.44,-0.05,"[-0.140, +0.030]",None,866,10.0,302.9
5,steering_gated,0.57,0.57,0.265,0.562,0.03,"[-0.030, +0.090]",6.0,3.0,NaN,NaN,,None,866,10.0,308.5
6,tact,0.54,0.54,0.147,0.562,0.00,"[+0.000, +0.000]",0.0,0.0,NaN,NaN,,None,866,10.0,325.6
7,lora,0.89,0.89,0.765,0.875,0.35,"[+0.240, +0.460]",37.0,2.0,0.88,0.39,"[+0.280, +0.500]",None,866,10.0,325.2


In [4]:
rows = []
for g in ("seen", "unseen"):
    for name, c in exp.state["transfer"][g]["comparisons"].items():
        rows.append({"group": g, "comparison": name, "diff": c["mean_diff"],
                     "ci": f"[{c['ci_low']:+.3f}, {c['ci_high']:+.3f}]", "excludes_zero": c["excludes_zero"]})
pd.DataFrame(rows)

,group,comparison,diff,ci,excludes_zero
0,seen,steering_vs_strong_prompt,0.06,"[-0.020, +0.150]",False
1,seen,steering_vs_restate,-0.07,"[-0.170, +0.040]",False
2,seen,steering_vs_explicit_state,-0.09,"[-0.190, +0.020]",False
3,seen,steering_vs_steering_gated,-0.01,"[-0.040, +0.020]",False
4,seen,steering_vs_tact,-0.01,"[-0.090, +0.080]",False
5,seen,steering_vs_lora,-0.38,"[-0.480, -0.280]",True
6,unseen,steering_vs_strong_prompt,0.05,"[-0.060, +0.160]",False
7,unseen,steering_vs_restate,0.05,"[-0.070, +0.170]",False
8,unseen,steering_vs_explicit_state,-0.33,"[-0.460, -0.210]",True
9,unseen,steering_vs_steering_gated,-0.05,"[-0.120, +0.010]",False


In [5]:
recs = {}
for g in ("seen", "unseen"):
    tasks = {t.task_id: t for t in exp.state["splits"][f"transfer:{g}"]}
    for m in ("baseline", "steering", "lora"):
        f = exp.dir / "decisions" / f"transfer-{g}__{m}.jsonl"
        df = pd.read_json(f, lines=True)
        df["need"] = [tasks[i].notes.split("need=")[1].split()[0] for i in df.task_id]
        recs[(g, m)] = df
pd.concat({k: v.groupby(["pair_type", "need", "branch"])["ok"].mean() for k, v in recs.items()}, axis=1).round(2)

seen                  unseen               
                          baseline steering  lora baseline steering  lora
pair_type need     branch                                                
changing  budget   0          0.90     0.70  1.00      NaN      NaN   NaN
                   1          0.60     0.70  1.00      NaN      NaN   NaN
          deadline 0          0.50     0.36  1.00      NaN      NaN   NaN
                   1          0.36     0.43  0.93      NaN      NaN   NaN
          quantity 0          0.76     0.65  0.88      NaN      NaN   NaN
                   1          0.41     0.41  0.59      NaN      NaN   NaN
invariant budget   0          0.00     0.33  1.00      NaN      NaN   NaN
                   1          0.00     0.33  1.00      NaN      NaN   NaN
          deadline 0          0.50     0.67  1.00      NaN      NaN   NaN
                   1          0.50     0.50  1.00      NaN      NaN   NaN
changing  rating   0           NaN      NaN   NaN     0.38     0.38  1.00
                   1           NaN      NaN   NaN     0.50     0.56  1.00
          warranty 0           NaN      NaN   NaN     0.56     0.33  0.67
                   1           NaN      NaN   NaN     0.44     0.67  0.89
invariant rating   0           NaN      NaN   NaN     0.58     0.50  0.92
                   1           NaN      NaN   NaN     0.67     0.58  0.92
          warranty 0           NaN      NaN   NaN     0.75     0.75  0.75
                   1           NaN      NaN   NaN     1.00     0.75  1.00

## Вывод → журнал

In [6]:
display(Markdown(exp.report(
    conclusion="""**Разведка, не подтверждающий результат.** Методы, выбранные на val в 04, без подбора — на 100 новых
задачах (seed 101): 50 со знакомыми типами требований (срок / количество / бюджет), 50 — с новыми (рейтинг,
гарантия); формат ответа общий. Test-сплит не использовался.

| метод | seen: успех, Δ к базе [95% CI] | unseen: успех, Δ к базе [95% CI] |
|---|---|---|
| база | 0.53 | 0.54 |
| steering | 0.52, −0.01 [−0.09, +0.08] | 0.52, −0.02 [−0.11, +0.06] |
| steering + гейт | 0.53, 0.00 | 0.57, +0.03 [−0.03, +0.09] |
| TACT | 0.53 (ни одного изменения) | 0.54 (ни одного изменения) |
| LoRA | 0.90, +0.37 [+0.26, +0.48] | 0.89, +0.35 [+0.24, +0.46] |
| явное состояние | 0.61, +0.08 [−0.01, +0.17] | 0.85, +0.31 [+0.20, +0.42] |
| пересказ / сильный промпт | 0.59 / 0.46 | 0.47 / 0.47 |

1. Прирост steering с val (+0.03) не воспроизвёлся даже на знакомых типах — скорее всего, шум отбора
среди 45 конфигураций на 29 задачах. Эпизоды: −0.04 / −0.05.
2. LoRA переносится почти без потерь: на новых типах прирост тот же, что на знакомых (+0.35 против +0.37),
специфичность 0.88–1.00, эпизоды +0.32 / +0.39. Она выучила общее правило «сверить ответ с требованием»,
а не шаблоны train — оговорка из 04 («LoRA учится на тех же шаблонах») снимается.
3. Явное состояние неожиданно сильно на новых типах (+0.31) и слабо на знакомых (+0.08) и на val (−0.07).
Неоднородность не объяснена, n мал; метод в 14 раз медленнее по задержке.
4. Адаптация TACT не изменила ни одного решения ни на одном наборе — в этой конфигурации она инертна
(оси на слоях 0, 24, 26, 27; проекции не выходят за полосу 2σ). Это ограничение адаптации, а не вывод о TACT.

Для H4 (разведочно): на условном срезе LoRA лучше steering и на знакомых, и на новых типах требований
(steering − LoRA = −0.38 и −0.37, интервалы исключают ноль); по переносимости (критерий 2604.14090) —
тоже LoRA. Ограничения: одна модель 1.7B, 50 задач на группу, выбор методов шёл на val.""",
    next_step="""Решение Павла по test: картина val и переноса согласована (steering не помогает, LoRA помогает).
Определить глобальный срез H4. Разведочно: почему явное состояние сильно на новых типах (по текстам
состояний); TACT — доля шагов вне полосы 2σ, до любых выводов о TACT.""",
)))

## 2026-10-20 01:48 · прогон `20261020-005120_transfer-qwen3-1.7b_c2afcb`

**Гипотеза:** explore · **Ноутбук:** notebooks/06_transfer.ipynb · **Модель:** Qwen/Qwen3-1.7B (mps, float16) · **Режим решения:** scored
**Стадии:** data, baseline, probe, vectors, sweep, fit, transfer · **git:** `71a4588`

**Зачем:** Перенос выбранных на val методов на новые типы требований (рейтинг, гарантия).

**Отличия от конфига по умолчанию:** `data.n_tasks` = 160; `eval.methods` = ['baseline', 'strong_prompt', 'restate', 'explicit_state', 'steering', 'steering_gated', 'tact', 'lora']; `model.name` = 'Qwen/Qwen3-1.7B'

**Результаты (val):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.621 | 0.526 | 0.400 | — |  | — | — | 0.586 | — | 831 | 10.000 | 400.600 |
| strong_prompt | 0.569 | 0.474 | 0.300 | -0.052 | [-0.138, +0.000] | 0.000 | 3.000 | 0.569 | -0.017 | 885 | 10.000 | 291.800 |
| restate | 0.638 | 0.526 | 0.400 | 0.017 | [-0.069, +0.103] | 4.000 | 3.000 | 0.586 | 0.000 | 1765 | 11.000 | 1596.100 |
| explicit_state | 0.552 | 0.474 | 0.100 | -0.069 | [-0.207, +0.069] | 6.000 | 10.000 | 0.466 | -0.121 | 1932 | 11.000 | 3443.700 |
| steering | 0.655 | 0.632 | 0.400 | 0.034 | [+0.000, +0.086] | 2.000 | 0.000 | 0.603 | 0.017 | 831 | 10.000 | 410.200 |
| steering_gated | 0.638 | 0.579 | 0.400 | 0.017 | [+0.000, +0.052] | 1.000 | 0.000 | 0.603 | 0.017 | 831 | 10.000 | 408.600 |
| tact | 0.621 | 0.526 | 0.400 | 0.000 | [+0.000, +0.000] | 0.000 | 0.000 | 0.586 | 0.000 | 831 | 10.000 | 369.600 |
| lora | 0.948 | 1.000 | 0.800 | 0.328 | [+0.190, +0.466] | 19.000 | 0.000 | 0.828 | 0.241 | 831 | 10.000 | 299.000 |

**Перенос, seen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.530 | 0.244 | 0.333 | — |  | — | — | 0.450 | — | 859 | 10.000 | 300.700 |
| strong_prompt | 0.460 | 0.195 | 0.333 | -0.070 | [-0.160, +0.010] | 3.000 | 10.000 | — | — | 913 | 10.000 | 303.700 |
| restate | 0.590 | 0.268 | 0.444 | 0.060 | [-0.030, +0.160] | 13.000 | 7.000 | — | — | 1850 | 11.000 | 2308.700 |
| explicit_state | 0.610 | 0.268 | 0.333 | 0.080 | [-0.010, +0.170] | 22.000 | 14.000 | — | — | 2018 | 11.000 | 4293.900 |
| steering | 0.520 | 0.220 | 0.444 | -0.010 | [-0.090, +0.080] | 6.000 | 7.000 | 0.410 | -0.040 | 859 | 10.000 | 300.000 |
| steering_gated | 0.530 | 0.220 | 0.333 | 0.000 | [-0.070, +0.070] | 5.000 | 5.000 | — | — | 859 | 10.000 | 300.300 |
| tact | 0.530 | 0.244 | 0.333 | 0.000 | [+0.000, +0.000] | 0.000 | 0.000 | — | — | 859 | 10.000 | 320.600 |
| lora | 0.900 | 0.756 | 1.000 | 0.370 | [+0.260, +0.480] | 37.000 | 0.000 | 0.770 | 0.320 | 859 | 10.000 | 317.700 |

**Перенос, unseen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | episode | ep_diff | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.540 | 0.147 | 0.562 | — |  | — | — | 0.490 | — | 866 | 10.000 | 310.200 |
| strong_prompt | 0.470 | 0.206 | 0.438 | -0.070 | [-0.150, +0.000] | 2.000 | 9.000 | — | — | 920 | 10.000 | 319.100 |
| restate | 0.470 | 0.147 | 0.438 | -0.070 | [-0.160, +0.020] | 5.000 | 12.000 | — | — | 1869 | 11.000 | 2350.900 |
| explicit_state | 0.850 | 0.618 | 0.938 | 0.310 | [+0.200, +0.420] | 34.000 | 3.000 | — | — | 2047 | 11.000 | 4374.200 |
| steering | 0.520 | 0.206 | 0.562 | -0.020 | [-0.110, +0.060] | 8.000 | 10.000 | 0.440 | -0.050 | 866 | 10.000 | 302.900 |
| steering_gated | 0.570 | 0.265 | 0.562 | 0.030 | [-0.030, +0.090] | 6.000 | 3.000 | — | — | 866 | 10.000 | 308.500 |
| tact | 0.540 | 0.147 | 0.562 | 0.000 | [+0.000, +0.000] | 0.000 | 0.000 | — | — | 866 | 10.000 | 325.600 |
| lora | 0.890 | 0.765 | 0.875 | 0.350 | [+0.240, +0.460] | 37.000 | 2.000 | 0.880 | 0.390 | 866 | 10.000 | 325.200 |

**Вердикты (автоматически, по правилам HYPOTHESES.md):**
- H1: ✔ support — лучший probe L16/post_message: AUROC 0.980, acc 0.882 при перестановке 0.573
- H2: ✔ support — различает (H1), но ошибается в 47% меняющих пар: чувствительность 0.526, специфичность 0.400
- H3: ? inconclusive (val) — на val: успех 0.621 → 0.655 при L16/pre_action/α=1.0; решает только test
- H4: ✘ against (val) — LoRA не хуже steering на условном срезе: steering − LoRA -0.293 [-0.431, -0.155]
- ALT: ? inconclusive (val) — то же качество, что у steering, но дороже: restate (steering − restate +0.017 [-0.052, +0.086]), но дороже: p50 1596 vs 410 мс; explicit_state (steering − explicit_state +0.103 [-0.034, +0.241]), но дороже: p50 3444 vs 410 мс

**Вывод:** **Разведка, не подтверждающий результат.** Методы, выбранные на val в 04, без подбора — на 100 новых
задачах (seed 101): 50 со знакомыми типами требований (срок / количество / бюджет), 50 — с новыми (рейтинг,
гарантия); формат ответа общий. Test-сплит не использовался.

| метод | seen: успех, Δ к базе [95% CI] | unseen: успех, Δ к базе [95% CI] |
|---|---|---|
| база | 0.53 | 0.54 |
| steering | 0.52, −0.01 [−0.09, +0.08] | 0.52, −0.02 [−0.11, +0.06] |
| steering + гейт | 0.53, 0.00 | 0.57, +0.03 [−0.03, +0.09] |
| TACT | 0.53 (ни одного изменения) | 0.54 (ни одного изменения) |
| LoRA | 0.90, +0.37 [+0.26, +0.48] | 0.89, +0.35 [+0.24, +0.46] |
| явное состояние | 0.61, +0.08 [−0.01, +0.17] | 0.85, +0.31 [+0.20, +0.42] |
| пересказ / сильный промпт | 0.59 / 0.46 | 0.47 / 0.47 |

1. Прирост steering с val (+0.03) не воспроизвёлся даже на знакомых типах — скорее всего, шум отбора
среди 45 конфигураций на 29 задачах. Эпизоды: −0.04 / −0.05.
2. LoRA переносится почти без потерь: на новых типах прирост тот же, что на знакомых (+0.35 против +0.37),
специфичность 0.88–1.00, эпизоды +0.32 / +0.39. Она выучила общее правило «сверить ответ с требованием»,
а не шаблоны train — оговорка из 04 («LoRA учится на тех же шаблонах») снимается.
3. Явное состояние неожиданно сильно на новых типах (+0.31) и слабо на знакомых (+0.08) и на val (−0.07).
Неоднородность не объяснена, n мал; метод в 14 раз медленнее по задержке.
4. Адаптация TACT не изменила ни одного решения ни на одном наборе — в этой конфигурации она инертна
(оси на слоях 0, 24, 26, 27; проекции не выходят за полосу 2σ). Это ограничение адаптации, а не вывод о TACT.

Для H4 (разведочно): на условном срезе LoRA лучше steering и на знакомых, и на новых типах требований
(steering − LoRA = −0.38 и −0.37, интервалы исключают ноль); по переносимости (критерий 2604.14090) —
тоже LoRA. Ограничения: одна модель 1.7B, 50 задач на группу, выбор методов шёл на val.

**Следующий шаг:** Решение Павла по test: картина val и переноса согласована (steering не помогает, LoRA помогает).
Определить глобальный срез H4. Разведочно: почему явное состояние сильно на новых типах (по текстам
состояний); TACT — доля шагов вне полосы 2σ, до любых выводов о TACT.
